# Week 10: More Stylometry

# Who wrote this boygenius song? 

In the Federalist Papers notebook we asked whether Hamilton or Madison wrote the disputed papers. Here we try the same methods song lyrics which are a very different kind of text.

[boygenius](https://en.wikipedia.org/wiki/Boygenius) is a band made up of three singer-songwriters who also have solo careers: **Julien Baker**, **Phoebe Bridgers** and **Lucy Dacus**. Their band songs are collaborations, but most have a member who leads the song. That gives us an attribution puzzle much like the Federalist one:

| Federalist Papers | boygenius |
|---|---|
| Papers known to be by Hamilton, Madison or Jay | Songs from each member's **solo albums** |
| 12 disputed papers | 22 **boygenius** songs |
| Each paper is about 2,000 to 3,000 words | Each song is about **200 words** |

Working with a small amount of data makes it harder to come to strong conclusions. Statistical methods work better with more data.

The dataset lives in the `lyrics_dataset/` folder. `songs.csv` lists every song with its artist, album and track number, and points to two text files per song:

* a **raw** file, which keeps section labels like `[Verse 1: Bridgers]` or `[Chorus: All]`, and
* a **clean** file, with those labels removed, so it holds only the lyrics.

In [ ]:
import pandas as pd

songs = pd.read_csv('lyrics_dataset/songs.csv')
songs

A few tracks are instrumentals with no (or almost no) lyrics, so we drop them. We also give the artists shorter names, using each member's surname.

In [ ]:
# A filter to select only songs with words
not_instrumental = songs["instrumental_note"] == False

# Make a copy of the data frame without instrumental songs
vocal_songs = songs[not_instrumental].copy()

# Shorten names for convenience
short_names = {"julien_baker": "Baker", "phoebe_bridgers": "Bridgers",
               "lucy_dacus": "Dacus", "boygenius": "boygenius"}
# A new pandas method to replace all occurences of the keys in short_names
# with their valuse
vocal_songs["artist"] = vocal_songs["artist"].map(short_names)

vocal_songs

The next step is to load the lyrics for each song into our dataframe.

In [ ]:

def read_lyrics(path):
    with open(f'lyrics_dataset/{path}', 'r') as f:
        return f.read()

vocal_songs["lyrics"] = vocal_songs["clean_file"].map(read_lyrics)

vocal_songs


In [ ]:
# Here's one song, so we can see what the clean lyrics look like
print(vocal_songs.loc[vocal_songs["title"] == "Emily I'm Sorry", "lyrics"].iloc[0])

Pandas has lots of different ways that we can look at the data. 

In [ ]:
# Organize the dataframe by a group
by_artist = vocal_songs.groupby("artist")
by_artist["title"].count()

We can even break down the number of songs by album and artist

In [ ]:
by_artist_and_album = vocal_songs.groupby(["artist", "album"])
by_artist_and_album["title"].count()

## Who sings lead? Building an answer key

With the Federalist Papers, historians had already formed a view about the disputed papers, and we compared our results against it. For boygenius we can build our own answer key from the **raw** lyric files, because their section labels say who sings each part:

In [ ]:
print(read_lyrics("raw/boygenius/the_record/06_not_strong_enough.txt")[:200])

A label like `[Chorus: Bridgers, Bridgers & Baker]` means Bridgers starts the chorus and Baker joins her partway through. We'll use a simple rule:

1. For each section, take the **first** singer named. If that's `All`, or there's no name, the section doesn't count for anyone.
2. Add up how many words each member leads.
3. If one member leads **at least 60%** of the words that were counted, call them the song's **lead**. Otherwise, call the song **Shared**.

Two caveats:

* We can't be sure that the person labeled as singing is the writer. We have some evidence from other writings that the boygenius member who brought a song to the band usually sings it, but the band says they write together, so "lead" is our best available stand-in for "main author".
* The 60% cutoff is a choice. A different cutoff would move a few songs between a member and Shared. 

The next code cell does this work.  It introduces the regular expression library,`re`.  Regular expressions give the the ability to define and apply patterns to extract text that matches these patterns. 

In this example below, the patterns in the text that we are trying to find are the ones that match labels like:

- `[Verse 1: Bridgers]`
- `[Chorus: Bridgers, Bridgers & Baker]`
- `[Refrain: All]`
- `[Verse 2: Julien Baker]`
- `[Outro: All]`

Can you see the pattern?


The other interesting pandas feature to point out is that we are writing a function below that will operate on one cell of our DataFrame and return a value. We can use the `pd.map()` method to create a new column by mapping the function to appropriate cell in each row.

In [ ]:
import re

members = ["Baker", "Bridgers", "Dacus"]


def lead_singer(raw_path):
    ''' Given a file containing the lyrics for a song (raw_path)
        - read the lyrics into a list of lines
        - each time we find a new label extract the first singer named
        and start counting words to attribute to that singer.
        - use the word counts for each singer to determine who led the song
    '''

    words_led = {"Baker": 0, "Bridgers": 0, "Dacus": 0}
    singer = None

    for line in read_lyrics(raw_path).splitlines():
        line = line.strip()

        # check if this is a line that matches the pattern
        label = re.fullmatch(r"\[(.*)\]", line)
        if label:
            # A new section, e.g. "Verse 1: Bridgers, All"
            singer = None
            if ":" in label.group(1):
                names = label.group(1).split(":", 1)[1]
                first = re.split(r"[,&]", names)[0]   # first singer named
                for m in members:
                    if m in first:
                        singer = m
        elif line and singer:
            # if this is a lyrics line, count the number of words
            # and this to the current lead singer
            words_led[singer] += len(line.split())

    # to whom did we attribute the most words?
    sum = 0
    best = "Baker"
    for artist in words_led:
        sum += words_led[artist]
        if words_led[artist] > words_led[best]:
            best = artist

    # Did artist with the most words have at least 60% of them?
    if words_led[best] >= 0.6 * sum:
        return best
    else:
        return "Shared"

bg = vocal_songs[vocal_songs["artist"] == "boygenius"].copy()
bg["lead"] = bg["raw_file"].map(lead_singer)
bg[["album", "title", "lead"]]

In [ ]:
bg["lead"].value_counts()

That gives us 16 songs with a clear lead and 6 shared songs. The shared songs are like the Federalist "Shared" papers: we can look at them, but there's no single right answer.



## Now we can start with the analysis

We will use NLTK (Natural Language Toolkit) to break the lyrics into tokens using the same technique that we did with the Federalist papers.NLTK splits the text into tokens, and we keep only tokens containing at least one letter, lowercased.

One thing that's different with lyrics: songs are full of contractions and casual spellings. NLTK splits *don't* into *do* + *n't*, *I'm* into *i* + *'m*, and *gonna* into *gon* + *na*. That's fine, as long as we treat every artist the same way.

In [ ]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

def get_words(text):
    words = []
    for token in nltk.word_tokenize(text):
        # Check each character: does the token contain at least one letter?
        # (This drops punctuation tokens like "," or "...")
        has_letter = False
        for character in token:
            if character.isalpha():
                has_letter = True
        # keep the token only if it has a letter
        if has_letter:
            words.append(token.lower())
    return words

vocal_songs["words"] = vocal_songs["lyrics"].map(get_words)
bg["words"] = vocal_songs.loc[bg.index, "words"]

print(get_words("I don't wanna be cool about it"))

bg

Now we combine each artist's songs into one long list of words, like `federalist_by_author` in the Federalist notebook.

In [ ]:
artists = ["Baker", "Bridgers", "Dacus", "boygenius"]

words_by_artist = {}
for artist in artists:
    words_by_artist[artist] = []
    # loop over just the song by "artist"
    artist_songs = vocal_songs[vocal_songs["artist"] == artist]
    for words in artist_songs["words"]:
        for word in words:
            words_by_artist[artist].append(word)

for artist in artists:
    print(artist, len(words_by_artist[artist]), "words")

For comparison: Hamilton alone wrote about 114,000 words of Federalist Papers. **Our whole lyrics dataset is smaller than that.** Every method below will be working with far less evidence.

## A First Measure: The Type-Token Ratio

As before, the **type-token ratio** is the number of distinct words (types) divided by the total number of words (tokens).

### Attempt 1: TTR for each artist's whole corpus

In [ ]:
# Each row in rows will have the number tokens, types and the TTR for one artist
rows = []
for artist in artists:
    # words is the list of tokens for an artist
    words = words_by_artist[artist]
    rows.append({
        "artist": artist,
        "songs": (vocal_songs["artist"] == artist).sum(),
        "tokens": len(words),
        "types": len(set(words)),
        "TTR": len(set(words)) / len(words),
    })

naive_ttr = pd.DataFrame(rows).set_index("artist")
naive_ttr.round(3)

As we saw last week, the ranking of TTR is just the ranking of corpus size turned upside down: Dacus has the most words and the lowest TTR; boygenius has the fewest and the highest. That tells us how much each artist wrote, not how they write.

We can see this clearly in the vocabulary curves.

In [ ]:
import matplotlib.pyplot as plt

colors = {"Baker": "#2a78d6", "Bridgers": "#eb6834", "Dacus": "#eda100",
          "boygenius": "#1baf7a", "Shared": "#e87ba4"}

fig, ax = plt.subplots(figsize=(9, 5))
for artist in artists:
    seen = set()
    types_so_far = []
    for word in words_by_artist[artist]:
        seen.add(word)
        types_so_far.append(len(seen))
    ax.plot(types_so_far, color=colors[artist], linewidth=2, label=artist)
    ax.annotate(artist, (len(types_so_far), types_so_far[-1]),
                xytext=(4, 0), textcoords="offset points", va="center", fontsize=9)

ax.set_xlabel("Tokens read so far")
ax.set_ylabel("Distinct types seen so far")
ax.set_title("Vocabulary growth: new words get rarer the longer you listen")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
plt.show()

### Attempt 2: Standardized TTR

We cut each artist's words into equal chunks and average the chunk TTRs. With the Federalist Papers we used 1,000-word chunks; here that would leave boygenius with only 4 chunks, so we use **500-word chunks** instead. (A TTR from 500-word chunks is higher than one from 1,000-word chunks, so don't compare these numbers with the Federalist ones.)

In [ ]:
CHUNK_SIZE = 500

# Return a list of ttrs for a document divided into chunks
def chunk_ttrs(words, size):
    ttrs = []
    for start in range(0, len(words) - size + 1, size):
        chunk = words[start:start + size]
        # The set function takes a list an removes all duplicates
        ttrs.append(len(set(chunk)) / size)
    return ttrs

rows = []
for artist in artists:
    ttrs = chunk_ttrs(words_by_artist[artist], CHUNK_SIZE)
    rows.append({
        "artist": artist,
        "chunks": len(ttrs),
        "STTR (mean)": sum(ttrs) / len(ttrs),
        "lowest chunk": min(ttrs),
        "highest chunk": max(ttrs),
    })

sttr = pd.DataFrame(rows).set_index("artist")
sttr.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for i, artist in enumerate(artists):
    ttrs = chunk_ttrs(words_by_artist[artist], CHUNK_SIZE)

    # Shift each dot a little left or right of the artist's position (i)
    # so the dots don't all sit on top of each other
    xs = []
    for j in range(len(ttrs)):
        xs.append(i + (j % 7 - 3) * 0.04)

    ax.scatter(xs, ttrs, color=colors[artist], s=30, alpha=0.7)
    mean = sum(ttrs) / len(ttrs)
    ax.plot([i - 0.25, i + 0.25], [mean, mean], color="black", linewidth=2)

ax.set_xticks(range(len(artists)))
# Label each artist with how many chunks they have
labels = []
for artist in artists:
    n_chunks = len(chunk_ttrs(words_by_artist[artist], CHUNK_SIZE))
    labels.append(f"{artist}\n({n_chunks} chunks)")
ax.set_xticklabels(labels)
ax.set_ylabel(f"TTR of a {CHUNK_SIZE}-token chunk")
ax.set_title(f"Each dot is one {CHUNK_SIZE}-word chunk; black bars are the averages")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
plt.show()

### What does this tell us?

* Once length is controlled, the three members are much closer together. Baker and Bridgers are nearly tied, and Dacus is a little lower.
* **Look at the spread.** Lyrics vary far more from chunk to chunk than the Federalist Papers did. Some Dacus chunks drop below 0.30, because a 500-word chunk might be mostly a single song whose chorus repeats over and over. This is to be expected since repetition is part of the genre so it has a big effect on TTR.
* boygenius songs land near the bottom, close to Dacus. Group songs often end in a chorus or outro that everyone sings several times, which pulls TTR down.

Let's look at more detailed features.



## Mendenhall's Characteristic Curves of Composition

We compare the **percentage** of each artist's words that have 1 letter, 2 letters, and so on.


In [ ]:
lengths = list(range(1, 13))

fig, ax = plt.subplots(figsize=(9, 5))
for artist in artists:
    words = words_by_artist[artist]

    # Start every length (1 to 12 letters) at a count of 0
    length_counts = {}
    for n in lengths:
        length_counts[n] = 0

    # Count how many words have each length.
    # Words longer than 12 letters are skipped (there are very few).
    for word in words:
        if len(word) in length_counts:
            length_counts[len(word)] += 1

    # Turn each count into a percentage of ALL the artist's words
    percentages = []
    for n in lengths:
        percentages.append(100 * length_counts[n] / len(words))

    ax.plot(lengths, percentages, marker="o", linewidth=2, color=colors[artist], label=artist)

ax.set_xlabel("Word length (letters)")
ax.set_ylabel("Percent of words")
ax.set_xticks(lengths)
ax.set_title("Word-length curves are very similar")
ax.legend(title="Artist", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
plt.show()

The curves are mostly on top of each other. Song lyrics are built from short, everyday words, and all three songwriters work in the same genre. (Notice that very few words are longer than 7 letters. The Federalist Papers had many more long words.)

The one visible difference is **Baker at 3 letters**: her curve is flat from 2 to 4 letters, while everyone else has a peak at 3. Which 3-letter words could cause that? (*you* and *the* are the most common ones)

The boygenius curve sits right among the three members, so word length gives us no way to tell who leads a band song.



## Second Stylometric Test: Burrows's Delta

Now the most widely used method. As before, we build a fingerprint from the relative frequencies of the most common words, convert to z-scores so every word gets an equal vote, and measure the average difference.

### A warm-up: *I*, *you* and *we*

With the Federalist Papers, Mosteller and Wallace's star word was *upon*. Song lyrics are about people, so pronouns are a natural place to look. Let's compute, for each song, how many times per 1,000 words it uses *i*, *you* and *we*, and then average by artist.

In [ ]:
def rate_per_1000(word, words):
    # Count the word and the total number of words in the song.
    word_count = words.count(word)
    total_words = len(words)
    return 1000 * word_count / total_words

# Calculate a rate for each word in each song.
for word in ["i", "you", "we"]:
    rates = []
    for song_words in vocal_songs["words"]:
        rate = rate_per_1000(word, song_words)
        rates.append(rate)
    vocal_songs[word] = rates

# Find the average rate for each artist and round the results.
by_artist = vocal_songs.groupby("artist")
mean_rates = by_artist[["i", "you", "we"]].mean()
mean_rates.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for artist in artists:
    s = vocal_songs[vocal_songs["artist"] == artist]
    ax.scatter(s["i"], s["you"], color=colors[artist], s=40, alpha=0.75, label=artist,
               marker="D" if artist == "boygenius" else "o")

ax.set_xlabel("'i' per 1,000 words")
ax.set_ylabel("'you' per 1,000 words")
ax.set_title("Each dot is one song")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(alpha=0.3)
plt.show()

There's a real pattern in the averages: Baker writes the most about *I* and the least about *you*, and she almost never uses *we*. Dacus uses *you* the most. Bridgers is in between.

But the dots are all mixed together on the scatter plot. A single 200-word song can easily be mostly about *you* or mostly about *I*, whoever wrote it. Compare that with *upon*, which split Hamilton and Madison cleanly. Unforunately, we don't have a single word that does that with the boygenious dataset, so we'll need the full fingerprint.

### Step 1: Choose the words for the fingerprint

We take the 30 most frequent words across the three members' solo songs.

In [ ]:
N_WORDS = 30

known = vocal_songs[vocal_songs["artist"].isin(members)]

known_words = []
for member in members:
    # extend is like append but adds the whole list to the end of known_words
    known_words.extend(words_by_artist[member])

# most_common gives (word, count) pairs; we only need the words
feature_words = []
for word, count in nltk.FreqDist(known_words).most_common(N_WORDS):
    feature_words.append(word)
print(feature_words)

These are almost all function words and pronouns, as we want. Notice the contraction pieces like *n't*, *'s* and *'m*: in lyrics, contractions are very common, and how often a writer uses them is itself a stylistic habit.

### Step 2: Build fingerprints

One fingerprint per member (from all their solo songs combined), and one per song.

In [ ]:
def fingerprint(words):
    counts = nltk.FreqDist(words)
    rates = {}
    for word in feature_words:
        rates[word] = counts[word] / len(words)
    return pd.Series(rates)

# One fingerprint per member, from all their solo songs combined.
member_rows = {}
for member in members:
    member_rows[member] = fingerprint(words_by_artist[member])
# .T flips (transposes) the table so that each member is a row
member_fingerprints = pd.DataFrame(member_rows).T

# One row per song (every song: solo and boygenius). The row label is the song's index.
song_rows = {}
for i, words in vocal_songs["words"].items():
    song_rows[i] = fingerprint(words)
song_fingerprints = pd.DataFrame(song_rows).T

(100 * member_fingerprints).round(2)

### Step 3: z-scores

We compute the mean and standard deviation of each word across all the **solo** songs, and convert every fingerprint to z-scores.

A positive z-score for means an artist uses that word more often than the average solo song; a negative score means they use it
less often. A score near zero means their usage is close to average.

Z-scores let us compare words on a common scale, even when some words are naturally much more frequent than others.

In [ ]:
word_means = song_fingerprints.loc[known.index].mean()
word_stds = song_fingerprints.loc[known.index].std()

def z_scores(fp):
    return (fp - word_means) / word_stds

member_z = z_scores(member_fingerprints)
member_z.round(2)

In [ ]:
# One panel per member, stacked so the same word lines up in every panel.
# sharex and sharey make all three panels use the same axes, so the bars
# can be compared directly.
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True, sharey=True)

for i, member in enumerate(members):
    ax = axes[i]
    ax.bar(feature_words, member_z.loc[member], color=colors[member])
    ax.axhline(0, color="grey", linewidth=1)   # z = 0 means "average use"
    ax.set_title(member, loc="left")
    ax.set_ylabel("z-score")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)

# Word labels go on the bottom panel only, turned so they don't overlap
axes[2].tick_params(axis="x", rotation=60)
fig.suptitle("Each member's fingerprint: above 0 = uses the word more than the average solo song")
plt.tight_layout()
plt.show()


Each member has a recognizable pattern. For example, Baker is high on *i*, *my* and *me* and low on *you*; Dacus is high on *you* and *your*. Notice how small these numbers are, though: none is bigger than ±0.5. Compare that with Jay in the Federalist notebook, whose z-scores went past ±2. **Measured against how much individual songs vary, the three songwriters differ only a little.**



### Step 4: Delta for each boygenius song

For each boygenius song, we compute one number per member: how far the song's word habits are from that member's. **The smaller the number, the closer the match.** The `delta` function does this in four moves:

1. **Turn the song into z-scores.** `z_scores(unknown_fp)` rescales the song's rate for each of the 30 feature words against the solo-song average for that word. A positive score means this song uses the word more than usual; a negative score means less. This puts common words like *the* and rarer words like *know* on the same scale.
2. **Subtract the member's z-scores.** `- member_z.loc[member]` gives 30 differences, one per word, between the song and the member's fingerprint.
3. **Ignore the direction.** `.abs()` drops the minus signs, because using a word too much and using it too little are both ways of differing.
4. **Average.** `.mean()` averages the 30 gaps into a single number. That number is Burrows's Delta.

The loop does this for every song and every member, filling in the Baker, Bridgers and Dacus columns. Then `idxmin` picks the member with the smallest Delta for each song and puts them in the `closest` column.

In short, Delta is **the average distance, word by word, between two fingerprints**, measured in z-scores so that every word counts equally.

In [ ]:
def delta(unknown_fp, member):
    return (z_scores(unknown_fp) - member_z.loc[member]).abs().mean()

results = bg[["title", "lead"]].copy()
for member in members:
    scores = []
    for i in bg.index:
        scores.append(delta(song_fingerprints.loc[i], member))
    results[member] = scores
results["closest"] = results[members].idxmin(axis=1)

results.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
ys = range(len(results))

for member in members:
    ax.scatter(results[member], ys, color=colors[member], s=60, label=member, zorder=3)

# a grey line spanning each song's three scores
for y, (_, row) in enumerate(results.iterrows()):
    ax.plot([row[members].min(), row[members].max()], [y, y],
            color="lightgrey", linewidth=2, zorder=1)

ax.set_yticks(list(ys))
# label each row with the song title and its lead singer
labels = []
for title, lead in zip(results["title"], results["lead"]):
    labels.append(f"{title} ({lead})")
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("Burrows's Delta (smaller = more similar)")
ax.set_title("Distance from each boygenius song to each member\n(the lead singer is in brackets)")
ax.legend(title="Member", frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=0.3)
plt.show()

In [ ]:
has_lead = results[results["lead"] != "Shared"]
matches = (has_lead["closest"] == has_lead["lead"]).sum()
print(f"Delta's closest member matches the lead singer for {matches} of {len(has_lead)} songs")
print(f"Random guessing among 3 members would get about {len(has_lead) / 3:.1f} right")

This is much less convincing than the Federalist result. Burrow's Delta does better than random guessing, but it gets a lot of songs wrong. And in the plot, the three dots for most songs are bunched close together, so even the "right" answers are often not by much.

But, we have evidence that our assumption that the person who sings the most in a song is not necessarily the writer of the song.


### Step 5: Group the songs, as the Federalist analysis did

In the Federalist notebook, the weakest attributions were near-ties for single papers, and the case was strongest for the papers **as a group**. Let's do the same here: combine all the boygenius songs that each member leads into one text, and compute Burrow's Delta for that.

In [ ]:
rows = []
for lead in ["Baker", "Bridgers", "Dacus", "Shared"]:
    group_words = []
    for words in bg.loc[bg["lead"] == lead, "words"]:
        group_words.extend(words)
    row = {"boygenius songs led by": lead, "words": len(group_words)}
    for member in members:
        row[member] = delta(fingerprint(group_words), member)
    rows.append(row)

delta_by_lead = pd.DataFrame(rows).set_index("boygenius songs led by")
delta_by_lead["closest"] = delta_by_lead[members].idxmin(axis=1)
delta_by_lead.round(3)

Grouped, Delta picks out the **Bridgers** songs clearly and the **Dacus** songs by a smaller margin. The **Baker** group is essentially a tie between Baker and Dacus. The evidence suggests that each member's style **does** carry over into the songs they lead, but even about 1,000 words gives only a modest signal.

### How much should we trust it? Testing on songs we know

As before, we take each solo song, pretend we don't know who wrote it, rebuild each member's fingerprint **without** that song (leave-one-out), and see whether Delta picks the right member.

In [ ]:
correct = 0
for i, song in known.iterrows():
    guesses = {}
    for member in members:
        others = known[(known["artist"] == member) & (known.index != i)]
        member_words = []
        for words in others["words"]:
            member_words.extend(words)
        guesses[member] = (z_scores(song_fingerprints.loc[i]) - z_scores(fingerprint(member_words))).abs().mean()
    if min(guesses, key=guesses.get) == song["artist"]:
        correct += 1

print(f"Correct: {correct} out of {len(known)} ({100 * correct / len(known):.0f}%)")
print(f"Random guessing would get about 33%")

For the Federalist Papers, leave-one-out testing got about **93%** right. Here it's below **50%**: better than chance, but wrong more often than right. That matches what we saw with the boygenius songs, and it tells us the problem is the **method on this kind of text**, not just the band songs being collaborative.

### What does this tell us?

* **Text length matters a lot.** Burrow's Delta works well on 2,000-word essays and not very well on 200-word songs. In a short song, a word like *the* might appear 5 times or 15 times largely by chance, so the fingerprint is mostly noise. Researchers usually say you need several thousand words for reliable attribution.
* **Grouping helps.** Once we combined songs into groups of about 750 to 1,650 words, Burrow's Delta mostly pointed to the right member, though often by small margins.
* **Genre matters.** All three members write first-person, conversational songs in the same genre. Hamilton and Madison had stable habits (like *upon*) that showed up in every paper; song lyrics are shaped as much by each song's subject, its *you*, and its chorus as by the writer's habits.
* **Our answer key is imperfect.** "Lead singer" is a stand-in for "main writer", and boygenius songs are genuinely collaborative. Some of Burrow's Delta's "mistakes" may reflect co-writing or mis-attribution.



**Things to try:**

* Change `N_WORDS` to 10, 50 or 100 and re-run from Step 1. Does the leave-one-out accuracy change?
* Change `CHUNK_SIZE` in the type-token section. How does the STTR change? Why?
* Try leave-one-out testing on **whole albums** instead of single songs: can Delta tell which member made an album?
* Change the 60% cutoff in `lead_singer()`. Which songs move between a member and Shared?
* Which words most separate Baker from Dacus? (Try `(member_z.loc["Baker"] - member_z.loc["Dacus"]).sort_values()`.)